In [2]:
import pymongo
import re

def update_city_with_timeline(city_name):
    client = pymongo.MongoClient("mongodb://localhost:27017/")
    db = client["Espana"]
    collection = db["ciudad"]

    doc = collection.find_one({"Ciudad": city_name})
    if not doc or 'Historia' not in doc:
        return

    # Sätze und Jahre extrahieren
    sentences = re.split(r'(?<=[.!?]) +', doc['Historia'])
    year_pattern = re.compile(r'\b(1\d{3}|20\d{2})\b')
    
    timeline_entries = []
    for s in sentences:
        match = year_pattern.search(s)
        if match:
            timeline_entries.append({
                "year": int(match.group(1)),
                "event": s.strip()
            })

    # Nach Jahr sortieren
    timeline_entries.sort(key=lambda x: x['year'])

    # Zurück in die DB schreiben (neues Feld 'Timeline')
    collection.update_one(
        {"Ciudad": city_name},
        {"$set": {"Timeline": timeline_entries}}
    )
    print(f"✅ Timeline für {city_name} wurde erfolgreich generiert und gespeichert.")

update_city_with_timeline("Manzanares")

✅ Timeline für Manzanares wurde erfolgreich generiert und gespeichert.


In [1]:
import pymongo

def find_oldest_history():
    client = pymongo.MongoClient("mongodb://localhost:27017/")
    db = client["Espana"]
    collection = db["ciudad"]

    # Suche alle Dokumente, die ein Timeline-Feld haben
    cities_with_timeline = collection.find({"Timeline": {"$exists": True, "$ne": []}})

    oldest_city = None
    min_year = 9999

    for city in cities_with_timeline:
        # Das erste Element der sortierten Timeline nehmen
        first_event = city['Timeline'][0]
        if first_event['year'] < min_year:
            min_year = first_event['year']
            oldest_city = {
                "name": city['Ciudad'],
                "year": min_year,
                "event": first_event['event']
            }

    if oldest_city:
        print(f"🏛️ Die historisch 'älteste' Stadt in deiner DB:")
        print(f"📍 Stadt: {oldest_city['name']}")
        print(f"📅 Jahr:  {oldest_city['year']}")
        print(f"📜 Event: {oldest_city['event']}")
    else:
        print("🔍 Keine Timelines gefunden. Hast du das Update-Skript schon laufen lassen?")

find_oldest_history()

🏛️ Die historisch 'älteste' Stadt in deiner DB:
📍 Stadt: Manzanares
📅 Jahr:  1239
📜 Event: Tras la victoriosa batalla de las Navas de Tolosa, en 1239 las órdenes militares se repartieron los vastos espacios conquistados a los musulmanes.
